# 당뇨 예측 - CatBoost 베이스 모델

- 타겟: `당뇨유병` (0: 없음 / 1: 있음)
- 모델: CatBoost Classifier
- 평가지표: Recall (주), F1-score, AUC-ROC

> **변경사항**
> - `class_weights`: `{0: 1.0, 1: neg/pos 비율}` — 양성에 집중 페널티
> - `iterations`: 500 + `early_stopping_rounds=50`
> - `eval_metric`: AUC

In [ ]:
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

matplotlib.rcParams["font.family"] = "DejaVu Sans"
warnings.filterwarnings("ignore")

# ── 경로 설정 ──────────────────────────────────────────────
INPUT_PATH   = '/Users/Jiyeon/Desktop/final_project/ML/data/x1_preprocessed.csv'
RANDOM_STATE = 42

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f'로드 완료 | shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')

## 2. 피처 / 타겟 분리 & 클래스 불균형 확인

In [ ]:
TARGET    = '당뇨유병'
DROP_COLS = ['고혈압유병', '당뇨유병', '이상지질혈증유병', '비만단계']

data = df.dropna(subset=[TARGET]).copy()
X = data.drop(columns=DROP_COLS)
y = data[TARGET].astype(int)

neg, pos = (y == 0).sum(), (y == 1).sum()
ratio = neg / pos

print(f'샘플 수: {len(y)}')
print(f'클래스 분포:\n{y.value_counts()}')
print(f'불균형 비율 (0:1): {neg}:{pos}  →  {ratio:.4f}')
print(f'\nclass_weights: {{0: 1.0, 1: {ratio:.4f}}}')

## 3. Stratified 5-Fold CV

In [ ]:
skf           = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proba     = np.zeros(len(y))
oof_pred      = np.zeros(len(y))
fold_scores   = []

print('=' * 55)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations            = 500,
        learning_rate         = 0.05,
        depth                 = 6,
        loss_function         = 'Logloss',
        eval_metric           = 'AUC',
        class_weights         = {0: 1.0, 1: ratio},  # 양성에 집중 페널티
        early_stopping_rounds = 50,
        random_seed           = RANDOM_STATE,
        verbose               = False,
        allow_writing_files   = False,
    )
    model.fit(Pool(X_tr, y_tr), eval_set=Pool(X_val, y_val))

    proba = model.predict_proba(X_val)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    oof_proba[val_idx] = proba
    oof_pred[val_idx]  = pred

    auc    = roc_auc_score(y_val, proba)
    f1     = f1_score(y_val, pred)
    recall = recall_score(y_val, pred)
    fold_scores.append({'fold': fold, 'auc': auc, 'f1': f1, 'recall': recall,
                        'best_iter': model.best_iteration_})
    print(f'  Fold {fold} | AUC: {auc:.4f} | F1: {f1:.4f} | Recall: {recall:.4f} '
          f'| best_iter: {model.best_iteration_}')

scores_df = pd.DataFrame(fold_scores)
print('=' * 55)
print(f"  평균   | AUC: {scores_df['auc'].mean():.4f} ± {scores_df['auc'].std():.4f} "
      f"| F1: {scores_df['f1'].mean():.4f} ± {scores_df['f1'].std():.4f} "
      f"| Recall: {scores_df['recall'].mean():.4f} ± {scores_df['recall'].std():.4f}")

## 4. OOF 전체 성능

In [ ]:
oof_auc    = roc_auc_score(y, oof_proba)
oof_f1     = f1_score(y, oof_pred)
oof_recall = recall_score(y, oof_pred)
oof_acc    = (oof_pred == y).mean()

print('=' * 45)
print('  [당뇨 CatBoost 베이스 모델 결과]')
print('=' * 45)
print(f'  Recall   : {oof_recall:.4f}')
print(f'  F1-score : {oof_f1:.4f}')
print(f'  AUC-ROC  : {oof_auc:.4f}')
print(f'  Accuracy : {oof_acc:.4f}')
print('=' * 45)
print()
print('[분류 리포트]')
print(classification_report(y, oof_pred, target_names=['정상(0)', '당뇨(1)']))

## 5. 혼동 행렬

In [ ]:
cm = confusion_matrix(y, oof_pred)
print(f'TN={cm[0,0]}  FP={cm[0,1]}')
print(f'FN={cm[1,0]}  TP={cm[1,1]}')

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['정상(0)', '당뇨(1)'])
disp.plot(cmap='Oranges')
plt.title('Confusion Matrix - 당뇨 CatBoost (OOF)')
plt.tight_layout()
plt.show()

## 6. ROC 커브

In [ ]:
RocCurveDisplay.from_predictions(y, oof_proba, name='CatBoost')
plt.title('ROC Curve - 당뇨 CatBoost (OOF)')
plt.tight_layout()
plt.show()

## 7. Feature Importance Top 15 (마지막 fold)

In [ ]:
fi = pd.DataFrame({
    'feature':    X.columns,
    'importance': model.get_feature_importance(),
}).sort_values('importance', ascending=False)

fi_top15 = fi.head(15)
plt.figure(figsize=(8, 6))
fi_top15[::-1].plot(x='feature', y='importance', kind='barh', color='darkorange', legend=False)
plt.title('Feature Importance Top 15 - 당뇨 CatBoost')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(fi_top15.to_string(index=False))